In [1]:
import sys
import hydra
import torch
from omegaconf import OmegaConf
from pathlib import Path

print(sys.executable)
print("Hydra OK")
print("OmegaConf OK")

# Path to src folder (where mhnfs/ lives)
project_root = Path.cwd()
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(project_root)
print(project_root / "src")

c:\Users\tom39\Documents\.venv\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


c:\Users\tom39\Documents\.venv\.venv\Scripts\python.exe
Hydra OK
OmegaConf OK
c:\Users\tom39\Desktop\MHNfs
c:\Users\tom39\Desktop\MHNfs\src


In [2]:
import inspect
import mhnfs.modules as mod
from mhnfs.modules import ContextModule


classes = [name for name, obj in inspect.getmembers(mod, inspect.isclass)]
print("Classes in mhnfs.modules:", classes)

print(inspect.signature(ContextModule.__init__))

Classes in mhnfs.modules: ['ContextModule', 'CrossAttentionModule', 'EncoderBlock', 'Hopfield', 'LayerNormalizingBlock', 'OmegaConf', 'partial']
(self, cfg: omegaconf.omegaconf.OmegaConf)


c:\Users\tom39\Desktop\MHNfs\src\mhnfs\modules.py:679: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path="../configs", config_name="cfg")
c:\Users\tom39\Desktop\MHNfs\src\mhnfs\modules.py:698: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path="../configs", config_name="cfg")
c:\Users\tom39\Desktop\MHNfs\src\mhnfs\modules.py:719: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path="../configs", config_name="cfg")


### Context Module

In [8]:
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial

PROJECT_ROOT = r"C:\Users\tom39\Desktop\MHNfs"
sys.path.append(PROJECT_ROOT)

from hopfield.my_hopfield import MyHopfield
#from mhnfs.modules import Hopfield # this is the original


In [9]:
# -------------------------------------------------
# Weight initialization
# -------------------------------------------------
def init_weights(module_type, module):
    if module_type == "linear" and isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

In [10]:
# -------------------------------------------------
# ContextModule
# -------------------------------------------------
class ContextModule(nn.Module):
    """
    Stable Context Module

    Features
    --------
    • Multi-step Hopfield retrieval
    • Top-k memory selection
    • Transformer-style FFN refinement
    • Gated residual updates
    """

    def __init__(self, cfg, top_k=32):
        super().__init__()

        dim = cfg.model.associationSpace_dim
        ffn_mult = 4

        self.num_steps = cfg.model.hopfield.num_steps
        self.top_k = top_k

        # -------------------------------------------------
        # Hopfield Memory
        # -------------------------------------------------
        self.hopfield = MyHopfield(
            input_size=dim,
            num_heads=cfg.model.hopfield.heads,
            init_beta=cfg.model.hopfield.beta,
            attn_dropout=cfg.model.hopfield.dropout,
        )

        self.hopfield.apply(partial(init_weights, "linear"))

        # -------------------------------------------------
        # Projections
        # -------------------------------------------------
        self.query_proj = nn.Linear(dim, dim)
        self.active_proj = nn.Linear(dim, dim)
        self.inactive_proj = nn.Linear(dim, dim)

        # -------------------------------------------------
        # Gates
        # -------------------------------------------------
        self.query_gate = nn.Parameter(torch.full((dim,), -0.5))
        self.support_gate = nn.Parameter(torch.full((dim,), -1.0))

        # -------------------------------------------------
        # Normalization
        # -------------------------------------------------
        self.pre_norm = nn.LayerNorm(dim)
        self.ffn_norm = nn.LayerNorm(dim)

        # -------------------------------------------------
        # Feed Forward Network
        # -------------------------------------------------
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * ffn_mult),
            nn.GELU(),
            nn.Linear(dim * ffn_mult, dim),
        )

    # -------------------------------------------------
    # L2 normalization
    # -------------------------------------------------
    def l2_norm(self, x):
        return F.normalize(x, dim=-1, eps=1e-8)

    # -------------------------------------------------
    # Top-K context selection
    # -------------------------------------------------
    def topk_context(self, query, context):

        """
        query:   [B,1,D]
        context: [Nc,D] or [B,Nc,D]
        return:  [B,top_k,D]
        """

        B = query.size(0)
        D = query.size(-1)

        if context.dim() == 2:
            context = context.unsqueeze(0).expand(B, -1, -1)

        elif context.dim() == 3:
            if context.size(0) == 1 and B > 1:
                context = context.expand(B, -1, -1)

            elif context.size(0) != B:
                raise ValueError(
                    f"Batch mismatch query={B} context={context.size(0)}"
                )

        query_norm = F.normalize(query, dim=-1)
        context_norm = F.normalize(context, dim=-1)

        sim = torch.bmm(
            query_norm,
            context_norm.transpose(1, 2)
        ).squeeze(1)

        k = min(self.top_k, context.size(1))

        _, idx = torch.topk(sim, k=k, dim=-1)

        topk_context = torch.gather(
            context,
            1,
            idx.unsqueeze(-1).expand(-1, -1, D),
        )

        return topk_context

    # -------------------------------------------------
    # Single retrieval step
    # -------------------------------------------------
    def retrieval_step(self, query, sa, si, context):

        # select top-k memory
        context_topk = self.topk_context(query, context)

        # projections
        q_proj = self.query_proj(query)
        sa_proj = self.active_proj(sa)
        si_proj = self.inactive_proj(si)

        # combine states
        s = torch.cat((q_proj, sa_proj, si_proj), dim=1)

        # Hopfield retrieval
        s_h = self.hopfield(
            query=s,
            key=context_topk,
            value=context_topk,
        )

        # split states
        q_h = s_h[:, 0:1]
        sa_h = s_h[:, 1:1 + sa_proj.shape[1]]
        si_h = s_h[:, 1 + sa_proj.shape[1]:]

        # gates
        q_gate = torch.sigmoid(self.query_gate).view(1, 1, -1)
        s_gate = torch.sigmoid(self.support_gate).view(1, 1, -1)

        # residual updates
        query = query + q_gate * (q_h - q_proj)
        sa = sa + s_gate * (sa_h - sa_proj)
        si = si + s_gate * (si_h - si_proj)

        return query, sa, si

    # -------------------------------------------------
    # Transformer FFN block
    # -------------------------------------------------
    def ffn_block(self, x):

        x_norm = self.ffn_norm(x)

        return x + self.ffn(x_norm)

    # -------------------------------------------------
    # Forward pass
    # -------------------------------------------------
    def forward(
        self,
        query,
        support_actives,
        support_inactives,
        context
    ):

        # PreNorm
        query = self.pre_norm(query)
        support_actives = self.pre_norm(support_actives)
        support_inactives = self.pre_norm(support_inactives)
        context = self.pre_norm(context)

        # multi-step retrieval
        for _ in range(self.num_steps):

            query, support_actives, support_inactives = self.retrieval_step(
                query,
                support_actives,
                support_inactives,
                context,
            )

            # FFN refinement
            query = self.ffn_block(query)
            support_actives = self.ffn_block(support_actives)
            support_inactives = self.ffn_block(support_inactives)

        # final normalization
        query = self.l2_norm(query)
        support_actives = self.l2_norm(support_actives)
        support_inactives = self.l2_norm(support_inactives)

        return query, support_actives, support_inactives

In [13]:
# =========================
# CONFIG
# =========================
from omegaconf import OmegaConf

cfg = OmegaConf.create({
    "model": {
        "associationSpace_dim": 1024,
        "hopfield": {
            "dim_QK": 512,
            "heads": 8,
            "dropout": 0.1,
            "beta": 2.0,
            "num_steps": 3       # <-- add this
        }
    }
})

# =========================
# CREATE MODULE
# =========================
context_module = ContextModule(cfg)

# =========================
# SYNTHETIC TEST DATA
# =========================
B = 8      # batch size
Na = 5     # active supports
Ni = 7     # inactive supports
Nc = 64    # context size
D = cfg.model.associationSpace_dim

query = torch.randn(B, 1, D, requires_grad=True)
support_actives = torch.randn(B, Na, D, requires_grad=True)
support_inactives = torch.randn(B, Ni, D, requires_grad=True)
context = torch.randn(1, Nc, D)

# =========================
# FORWARD PASS
# =========================
q_out, sa_out, si_out = context_module(
    query,
    support_actives,
    support_inactives,
    context
)

# =========================
# SHAPE CHECK
# =========================
print("=== SHAPES ===")
print("Query:", q_out.shape)
print("Actives:", sa_out.shape)
print("Inactives:", si_out.shape)

# =========================
# NaN CHECK
# =========================
print("\n=== NaN CHECK ===")
print("Query:", torch.isnan(q_out).any().item())
print("Actives:", torch.isnan(sa_out).any().item())
print("Inactives:", torch.isnan(si_out).any().item())

# =========================
# CHANGE MAGNITUDE
# =========================
def change_mag(a, b):
    return (a - b).norm(dim=-1).mean()

print("\n=== EMBEDDING CHANGE MAGNITUDE ===")
print("Query:", change_mag(query.detach(), q_out.detach()).item())
print("Actives:", change_mag(support_actives.detach(), sa_out.detach()).item())
print("Inactives:", change_mag(support_inactives.detach(), si_out.detach()).item())

# =========================
# GRADIENT FLOW CHECK
# =========================
loss = q_out.mean() + sa_out.mean() + si_out.mean()
loss.backward()

print("\n=== GRADIENT NORMS ===")
print("Query grad:", query.grad.norm().item())
print("Actives grad:", support_actives.grad.norm().item())
print("Inactives grad:", support_inactives.grad.norm().item())

# =========================
# CONTEXT SIMILARITY
# =========================
def context_similarity(q, context):
    q_mean = q.mean(dim=0)
    c_mean = context.mean(dim=1)
    return F.cosine_similarity(q_mean, c_mean)

before = context_similarity(query.detach(), context.detach())
after = context_similarity(q_out.detach(), context.detach())

print("\n=== CONTEXT SIMILARITY ===")
print("Before:", before.item())
print("After:", after.item())

# ============================================================
# HYPERPARAMETER SENSITIVITY
# ============================================================

print("\n\n==============================")
print("BETA SWEEP")
print("==============================")

for beta in [0.1, 0.3, 0.5, 1.0, 2.0]:
    cfg.model.hopfield.beta = beta
    context_module = ContextModule(cfg)

    q_out, _, _ = context_module(query.detach(), support_actives.detach(), support_inactives.detach(), context)

    movement = (q_out - query.detach()).norm(dim=-1).mean()
    print(f"beta={beta:.2f} | query movement={movement.item():.4f}")


print("\n\n==============================")
print("HEADS SWEEP")
print("==============================")

for heads in [1, 2, 4, 8]:
    cfg.model.hopfield.heads = heads
    context_module = ContextModule(cfg)

    q_out, _, _ = context_module(query.detach(), support_actives.detach(), support_inactives.detach(), context)

    movement = (q_out - query.detach()).norm(dim=-1).mean()
    print(f"heads={heads} | query movement={movement.item():.4f}")


print("\n\n==============================")
print("dim_QK SWEEP")
print("==============================")

for dim in [128, 256, 512, 1024]:
    cfg.model.hopfield.dim_QK = dim
    context_module = ContextModule(cfg)

    q_out, _, _ = context_module(query.detach(), support_actives.detach(), support_inactives.detach(), context)

    movement = (q_out - query.detach()).norm(dim=-1).mean()
    print(f"dim_QK={dim} | query movement={movement.item():.4f}")

# =========================
# SYNTHETIC DATA
# =========================
B = 16
Na = 6
Ni = 6
Nc = 64
D = cfg.model.associationSpace_dim

query = torch.randn(B, 1, D)
support_actives = torch.randn(B, Na, D)
support_inactives = torch.randn(B, Ni, D)
context = torch.randn(1, Nc, D)

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def cosine_to_set(query, support):
    """
    query: [B,1,D]
    support: [B,N,D]
    returns: mean cosine similarity per batch
    """
    q = query.squeeze(1)  # [B,D]
    sims = []

    for i in range(support.shape[1]):
        sims.append(F.cosine_similarity(q, support[:, i, :], dim=-1))

    sims = torch.stack(sims, dim=1)  # [B,N]
    return sims.mean(dim=1)  # [B]

def separability_score(sim_actives, sim_inactives):
    """
    Higher = better separation
    """
    return (sim_actives - sim_inactives).mean()

def margin_distribution(sim_actives, sim_inactives):
    return sim_actives - sim_inactives

# ============================================================
# BEFORE CONTEXT
# ============================================================

sim_active_before = cosine_to_set(query, support_actives)
sim_inactive_before = cosine_to_set(query, support_inactives)

sep_before = separability_score(sim_active_before, sim_inactive_before)
margin_before = margin_distribution(sim_active_before, sim_inactive_before)

print("====== BEFORE CONTEXT ======")
print("Similarity to actives:", sim_active_before.mean().item())
print("Similarity to inactives:", sim_inactive_before.mean().item())
print("Separability score:", sep_before.item())
print("Margin mean:", margin_before.mean().item())
print("Margin std:", margin_before.std().item())

# ============================================================
# AFTER CONTEXT
# ============================================================

q_out, sa_out, si_out = context_module(
    query,
    support_actives,
    support_inactives,
    context
)

sim_active_after = cosine_to_set(q_out, sa_out)
sim_inactive_after = cosine_to_set(q_out, si_out)

sep_after = separability_score(sim_active_after, sim_inactive_after)
margin_after = margin_distribution(sim_active_after, sim_inactive_after)

print("\n====== AFTER CONTEXT ======")
print("Similarity to actives:", sim_active_after.mean().item())
print("Similarity to inactives:", sim_inactive_after.mean().item())
print("Separability score:", sep_after.item())
print("Margin mean:", margin_after.mean().item())
print("Margin std:", margin_after.std().item())

# ============================================================
# IMPROVEMENT METRICS
# ============================================================

print("\n====== IMPROVEMENT ======")

print("Separability gain:", (sep_after - sep_before).item())
print("Active similarity change:", (sim_active_after.mean() - sim_active_before.mean()).item())
print("Inactive similarity change:", (sim_inactive_after.mean() - sim_inactive_before.mean()).item())

effect_size = (margin_after.mean() - margin_before.mean()) / (margin_before.std() + 1e-8)
print("Effect size (Cohen-like):", effect_size.item())

# ============================================================
# BETA SWEEP — separation sensitivity
# ============================================================

print("\n\n====== BETA SWEEP (SEPARABILITY) ======")

for beta in [0.1, 0.3, 0.5, 1.0, 2.0]:
    cfg.model.hopfield.beta = beta
    context_module = ContextModule(cfg)

    q_out, sa_out, si_out = context_module(
        query,
        support_actives,
        support_inactives,
        context
    )

    sim_active_after = cosine_to_set(q_out, sa_out)
    sim_inactive_after = cosine_to_set(q_out, si_out)

    sep_after = separability_score(sim_active_after, sim_inactive_after)

    print(f"beta={beta:.2f} | separability={sep_after.item():.6f}")

# ============================================================
# Gate
# ============================================================

print("Query gate:", torch.sigmoid(context_module.query_gate).mean().item())
print("Support gate:", torch.sigmoid(context_module.support_gate).mean().item())

# ============================================================
# Learned Beta
# ============================================================

beta = context_module.hopfield.beta.detach().cpu()

print("Learned beta (mean):", beta.mean().item())

# convert tensor -> python list (no numpy needed)
print("Learned beta per head:", beta.tolist())

=== SHAPES ===
Query: torch.Size([8, 1, 1024])
Actives: torch.Size([8, 5, 1024])
Inactives: torch.Size([8, 7, 1024])

=== NaN CHECK ===
Query: False
Actives: False
Inactives: False

=== EMBEDDING CHANGE MAGNITUDE ===
Query: 31.32324981689453
Actives: 31.178211212158203
Inactives: 31.179885864257812

=== GRADIENT NORMS ===
Query grad: 0.00017068081069737673
Actives grad: 7.905913662398234e-05
Inactives grad: 6.720855890307575e-05

=== CONTEXT SIMILARITY ===
Before: -0.05799475684762001
After: -0.00016653351485729218


BETA SWEEP
beta=0.10 | query movement=31.3278
beta=0.30 | query movement=31.3276
beta=0.50 | query movement=31.3263
beta=1.00 | query movement=31.3285
beta=2.00 | query movement=31.3245


HEADS SWEEP
heads=1 | query movement=31.3235
heads=2 | query movement=31.3310
heads=4 | query movement=31.3310
heads=8 | query movement=31.3301


dim_QK SWEEP
dim_QK=128 | query movement=31.3296
dim_QK=256 | query movement=31.3239
dim_QK=512 | query movement=31.3274
dim_QK=1024 | query mo

In [113]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def cosine_to_set(query, support):
    """
    query: [B,1,D]
    support: [B,N,D]
    returns: mean cosine similarity per batch
    """
    q = query.squeeze(1)  # [B,D]
    sims = []

    for i in range(support.shape[1]):
        sims.append(F.cosine_similarity(q, support[:, i, :], dim=-1))

    sims = torch.stack(sims, dim=1)  # [B,N]
    return sims.mean(dim=1)  # [B]

def separability_score(sim_actives, sim_inactives):
    """
    Higher = better separation
    """
    return (sim_actives - sim_inactives).mean()

def margin_distribution(sim_actives, sim_inactives):
    return sim_actives - sim_inactives


# ============================================================
# BEFORE CONTEXT
# ============================================================

sim_active_before = cosine_to_set(query, support_actives)
sim_inactive_before = cosine_to_set(query, support_inactives)

sep_before = separability_score(sim_active_before, sim_inactive_before)
margin_before = margin_distribution(sim_active_before, sim_inactive_before)

print("====== BEFORE CONTEXT ======")
print("Similarity to actives:", sim_active_before.mean().item())
print("Similarity to inactives:", sim_inactive_before.mean().item())
print("Separability score:", sep_before.item())
print("Margin mean:", margin_before.mean().item())
print("Margin std:", margin_before.std().item())


# ============================================================
# AFTER CONTEXT
# ============================================================

q_out, sa_out, si_out = context_module(
    query,
    support_actives,
    support_inactives,
    context
)

sim_active_after = cosine_to_set(q_out, sa_out)
sim_inactive_after = cosine_to_set(q_out, si_out)

sep_after = separability_score(sim_active_after, sim_inactive_after)
margin_after = margin_distribution(sim_active_after, sim_inactive_after)

print("\n====== AFTER CONTEXT ======")
print("Similarity to actives:", sim_active_after.mean().item())
print("Similarity to inactives:", sim_inactive_after.mean().item())
print("Separability score:", sep_after.item())
print("Margin mean:", margin_after.mean().item())
print("Margin std:", margin_after.std().item())


# ============================================================
# IMPROVEMENT METRICS
# ============================================================

print("\n====== IMPROVEMENT ======")

print("Separability gain:", (sep_after - sep_before).item())
print("Active similarity change:", (sim_active_after.mean() - sim_active_before.mean()).item())
print("Inactive similarity change:", (sim_inactive_after.mean() - sim_inactive_before.mean()).item())

effect_size = (margin_after.mean() - margin_before.mean()) / (margin_before.std() + 1e-8)
print("Effect size (Cohen-like):", effect_size.item())


# ============================================================
# BETA SWEEP — separation sensitivity
# ============================================================

print("\n\n====== BETA SWEEP (SEPARABILITY) ======")

for beta in [0.1, 0.3, 0.5, 1.0, 2.0]:
    cfg.model.hopfield.beta = beta
    context_module = ContextModule(cfg)

    q_out, sa_out, si_out = context_module(
        query,
        support_actives,
        support_inactives,
        context
    )

    sim_active_after = cosine_to_set(q_out, sa_out)
    sim_inactive_after = cosine_to_set(q_out, si_out)

    sep_after = separability_score(sim_active_after, sim_inactive_after)

    print(f"beta={beta:.2f} | separability={sep_after.item():.6f}")


====== BEFORE CONTEXT ======
Similarity to actives: 0.0007955748005770147
Similarity to inactives: 0.010871639475226402
Separability score: -0.010076065547764301
Margin mean: -0.010076065547764301
Margin std: 0.016991551965475082



====== AFTER CONTEXT ======
Similarity to actives: 0.028680015355348587
Similarity to inactives: 0.03215736895799637
Separability score: -0.0034773547668009996
Margin mean: -0.0034773547668009996
Margin std: 0.014489923603832722

====== IMPROVEMENT ======
Separability gain: 0.006598711013793945
Active similarity change: 0.02788444049656391
Inactive similarity change: 0.021285729482769966
Effect size (Cohen-like): 0.38835224509239197


====== BETA SWEEP (SEPARABILITY) ======
beta=0.10 | separability=-0.007793
beta=0.30 | separability=-0.006288
beta=0.50 | separability=-0.010156
beta=1.00 | separability=-0.007648
beta=2.00 | separability=-0.001592


Task:

1. Add learnable gating ✅
2. Normalize embeddings ✅
3. Separate context projections per type ✅
4. Add learnable beta ✅
5. FFN after retrieval ✅
6. Multi-step retrieval ✅
7. Top-k memory ✅

### Original Code

##### Context Module

In [ ]:
class ContextModule(nn.Module):
    """
    Allows for mutual information sharing.
    Enriches the query and support set embeddings with context by associating a query or
    support set molecule with the context set, i.e., large set of training molecules:
    - The context set can be seen as an external memory
    - For a given molecule embedding, a Modern Hopfield Network retrieves a representa-
      tion from the external memory

    Since we have to retrieve representations for all query and support set molecules we
    stack all embeddings together and perform a "batch-retrieval".
    """

    def __init__(self, cfg: OmegaConf):
        super(ContextModule, self).__init__()

        self.cfg = cfg

        self.hopfield = Hopfield(
            input_size=self.cfg.model.associationSpace_dim,
            hidden_size=cfg.model.hopfield.dim_QK,
            stored_pattern_size=self.cfg.model.associationSpace_dim,
            pattern_projection_size=self.cfg.model.associationSpace_dim,
            output_size=self.cfg.model.associationSpace_dim,
            num_heads=self.cfg.model.hopfield.heads,
            scaling=self.cfg.model.hopfield.beta,
            dropout=self.cfg.model.hopfield.dropout,
        )

        # Initialization
        hopfield_initialization = partial(init_weights, "linear")
        self.hopfield.apply(hopfield_initialization)

    def forward(
        self,
        query_embedding: torch.Tensor,
        support_actives_embedding: torch.Tensor,
        support_inactives_embedding: torch.Tensor,
        context_set_embedding: torch.Tensor,
    ) -> tuple:
        """
        inputs:
        - query; torch.tensor;
          dim: [batch-size, 1, initial-embedding-dimension]
            * e.g.: [512, 1, 1024]
            * initial-embedding-dimension: defined by molecule encoder block
        - active support set molecules; torch.tensor;
          dim: [batch-size, active-padding-dim, initial-embedding-dimension]
          * e.g.: [512, 9, 1024]
        - inactive support set molecules; torch.tensor;
          dim: [batch-size, inactive-padding-dim, initial-embedding-dimension]
          * e.g.: [512, 11, 1024]
        - context set molecules; torch.tensor;
          dim: [1, number-of-context-molecules, initial-embedding-dimension]
          * e.g.: [1, 512, 1024]

        return:
        tuple which includes the updated representations for query, active, and inactive
        support set molecules:
        (query, active support set molecules, inactive support set molecules)
        """
        # Stack embeddings together to perform a "batch-retrieval"
        s = torch.cat(
            (query_embedding, support_actives_embedding, support_inactives_embedding),
            dim=1,
        )
        s_flattend = s.reshape(1, s.shape[0] * s.shape[1], s.shape[2])

        # Retrieval
        s_h = self.hopfield((context_set_embedding, s_flattend, context_set_embedding))

        # Combine retrieval with skip connection
        s_updated = s_flattend + s_h
        s_updated_inputShape = s_updated.reshape(
            s.shape[0], s.shape[1], s.shape[2]
        )  # reshape tensor back to input shape

        query_embedding = s_updated_inputShape[:, 0, :]
        query_embedding = torch.unsqueeze(query_embedding, 1)

        # Split query, active and inactive support set embeddings
        padding_size_actives = support_actives_embedding.shape[1]

        support_actives_embedding = s_updated_inputShape[
            :, 1 : (padding_size_actives + 1), :
        ]
        support_inactives_embedding = s_updated_inputShape[
            :, (padding_size_actives + 1) :, :
        ]

        return query_embedding, support_actives_embedding, support_inactives_embedding